# 01 — Core Language: Collections, Functions, OOP

**Overview:** This notebook covers Python collections, comprehensions, functions
(`*args`/`**kwargs`, `lambda`, `map`/`filter`/`zip`/`enumerate`), and OOP (classes, inheritance
vs composition, `@dataclass`, magic methods).

**Required libraries:** none beyond the standard library.

Run top to bottom. Each section: concept → small example → output → interpretation → connection
to ML, closing with exercises, interview questions, and a summary.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **OOP** | Object-Oriented Programming | Style that bundles data and behaviour into classes/objects (inheritance, encapsulation). |
| **ML** | Machine Learning | Learning patterns from data to make predictions, instead of hand-coding rules. |

## 0. Environment

In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
SEED = 42


Python: 3.12.5 (main, Aug 29 2025, 10:17:29) [Clang 17.0.0 (clang-1700.0.13.5)]
Platform: macOS-26.5.1-arm64-arm-64bit


## 1. Collections: list, tuple, set, dict

💡 **Key Idea:** list/dict are mutable, tuple is immutable, set holds unique unordered items.
Indexing and slicing (`a[1:3]`, `a[::-1]`) work the same way across sequences.

In [2]:
a_list = [1, 2, 3, 4, 5]
a_tuple = (1, 2, 3)
a_set = {1, 2, 2, 3}          # duplicates collapse
a_dict = {"lr": 0.001, "batch_size": 32}

print("list:", a_list, "slice [1:3]:", a_list[1:3], "reversed:", a_list[::-1])
print("tuple:", a_tuple)
print("set (dedup):", a_set)
print("dict:", a_dict)

assert a_list[1:3] == [2, 3]
assert a_list[::-1] == [5, 4, 3, 2, 1]
assert len(a_set) == 3


list: [1, 2, 3, 4, 5] slice [1:3]: [2, 3] reversed: [5, 4, 3, 2, 1]
tuple: (1, 2, 3)
set (dedup): {1, 2, 3}
dict: {'lr': 0.001, 'batch_size': 32}


🧠 **Remember:** `a_tuple[0] = 99` raises `TypeError` — tuples are immutable, which is why
they're safe to use as dict keys and function default arguments.

⚠️ **Common Mistake:** `def f(bucket=[])` — a mutable default is created **once** and shared
across every call. Demonstrated below.

In [3]:
def bad_append(item, bucket=[]):
    bucket.append(item)
    return bucket

first_call = bad_append("a")
second_call = bad_append("b")
print("first_call:", first_call)
print("second_call:", second_call, "(unexpectedly also contains 'a'!)")

def good_append(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print("good_append:", good_append("a"))
print("good_append:", good_append("b"))

assert second_call == ["a", "b"]
assert good_append("c") == ["c"]


first_call: ['a', 'b']
second_call: ['a', 'b'] (unexpectedly also contains 'a'!)
good_append: ['a']
good_append: ['b']


## 2. Comprehensions

💡 **Key Idea:** comprehensions replace build-and-append loops.

In [4]:
squares = [x**2 for x in range(10) if x % 2 == 0]
label_set = {len(w) for w in ["cat", "dog", "fox", "wolf"]}
lookup = {w: len(w) for w in ["cat", "dog", "wolf"]}

print("even squares:", squares)
print("unique word lengths:", label_set)
print("word -> length:", lookup)

assert squares == [0, 4, 16, 36, 64]
assert lookup == {"cat": 3, "dog": 3, "wolf": 4}


even squares: [0, 4, 16, 36, 64]
unique word lengths: {3, 4}
word -> length: {'cat': 3, 'dog': 3, 'wolf': 4}


## 3. Functions: *args, **kwargs, lambda, map/filter/zip/enumerate

🎯 **Interview Tip:** `*args`/`**kwargs` is exactly how `nn.Module.forward(*args, **kwargs)`
and optimizer constructors stay flexible without breaking every caller when a new parameter
is added.

In [5]:
def describe_call(*args, **kwargs):
    return f"positional={args}, keyword={kwargs}"

print(describe_call(1, 2, lr=0.01, momentum=0.9))

square = lambda x: x**2
nums = [1, 2, 3, 4]
names = ["a", "b", "c", "d"]

print("map square:", list(map(square, nums)))
print("filter even:", list(filter(lambda x: x % 2 == 0, nums)))
print("zip:", list(zip(nums, names)))
print("enumerate:", list(enumerate(names, start=1)))

assert list(map(square, nums)) == [1, 4, 9, 16]
assert list(filter(lambda x: x % 2 == 0, nums)) == [2, 4]


positional=(1, 2), keyword={'lr': 0.01, 'momentum': 0.9}
map square: [1, 4, 9, 16]
filter even: [2, 4]
zip: [(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]
enumerate: [(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 4. OOP: classes, inheritance vs composition, dataclasses, magic methods

🧠 **Remember:** prefer composition ("has-a") over inheritance ("is-a") for model components —
it's easier to swap one part (e.g. an optimizer) without an inheritance hierarchy fighting you.

In [6]:
from dataclasses import dataclass, field

@dataclass
class TrainingExample:
    text: str
    label: int

    def __len__(self):
        return len(self.text)

ex = TrainingExample(text="great movie", label=1)
print(ex)                 # dataclass auto-generates __repr__
print("len(ex):", len(ex))

assert len(ex) == len("great movie")
assert ex.label == 1


TrainingExample(text='great movie', label=1)
len(ex): 11


In [7]:
# Composition: a Trainer HAS-A model and an optimizer, rather than inheriting from both.
class LinearModel:
    def __init__(self, weight: float):
        self.weight = weight
    def predict(self, x: float) -> float:
        return self.weight * x

class SGDOptimizer:
    def __init__(self, lr: float):
        self.lr = lr
    def step(self, model: LinearModel, grad: float):
        model.weight -= self.lr * grad

class Trainer:
    def __init__(self, model: LinearModel, optimizer: SGDOptimizer):
        self.model = model          # composition, not inheritance
        self.optimizer = optimizer

trainer = Trainer(LinearModel(weight=0.5), SGDOptimizer(lr=0.1))
trainer.optimizer.step(trainer.model, grad=2.0)
print("weight after one step:", trainer.model.weight)

assert abs(trainer.model.weight - 0.3) < 1e-9


weight after one step: 0.3


In [8]:
# Magic methods: __repr__, __len__, __getitem__, __iter__ — this is exactly the contract a
# PyTorch Dataset expects (__len__ + __getitem__).
class ToyDataset:
    def __init__(self, data: list[float]):
        self._data = data

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> float:
        return self._data[idx]

    def __iter__(self):
        return iter(self._data)

    def __repr__(self) -> str:
        return f"ToyDataset(n={len(self)})"

ds = ToyDataset([0.1, 0.2, 0.3])
print(ds, "len:", len(ds), "ds[1]:", ds[1], "list(ds):", list(ds))

assert len(ds) == 3
assert ds[1] == 0.2
assert list(ds) == [0.1, 0.2, 0.3]


ToyDataset(n=3) len: 3 ds[1]: 0.2 list(ds): [0.1, 0.2, 0.3]


## 5. Connection to what's next

`ToyDataset` above already looks like a PyTorch `Dataset`. In `02_numpy` we replace Python lists
with `ndarray`s so these same operations vectorize instead of looping in pure Python.

## 6. Interview questions
1. Why is a tuple hashable but a list is not, and why does that matter for dict keys?
2. When would you choose composition over inheritance for a model component?
3. What's the difference between `__repr__` and `__str__`?
4. Why does `def f(x=[])` cause bugs across repeated calls?
5. What does `__getitem__` + `__len__` let a class do that it otherwise couldn't?

## 7. Exercises
1. Add a `__iter__` to `TrainingExample` that is meaningless on its own — instead, build a
   `Batch` dataclass holding a `list[TrainingExample]` and give it `__len__`/`__getitem__`.
2. Rewrite `describe_call` so it validates that `kwargs` only contains keys from an allowed set.
3. Using `zip` and a dict comprehension, build a mapping from word → its length for a list of words,
   without using a `for` loop body (comprehension only).

## 8. Summary

✅ Collections (list/tuple/set/dict), comprehensions, flexible function signatures
(`*args`/`**kwargs`), and OOP (composition + dataclasses + magic methods) are the vocabulary the
rest of this curriculum assumes you have. Next: `02_generators_decorators_contextmanagers.ipynb`.